# SACRA — Surxondaryo bug'doychiligida iqlim riski va indeksli sug'urta

Ushbu daftar dissertatsiyaning empirik qismini boshidan oxirigacha qayta hisoblaydi.
Har bir katak alohida bosqichga to'g'ri keladi va o'z natijasini chop etadi.

**Guvohnoma:** № DGU 60955 | **Muallif:** A.S. Abdullayev | **Versiya:** 1.1.0

## 0. Tayyorgarlik

In [ ]:
!git clone -q https://github.com/alishersadullayevich97-create/sacra-index-insurance.git
%cd sacra-index-insurance
!pip install -q -r requirements.txt
%cd src

## 1–3. Ma'lumot, trend va anomaliya

Texnologik trend ajratilmasa, u tebranish sifatida talqin qilinib, risk bahosini buzadi.

In [ ]:
from s01_data import load_province, load_districts, load_climate, load_station
from s02_detrend import detrend_series, detrend_panel

v = load_province()
res = detrend_series(v.year, v.production_t)
v = v.assign(anom_abs=res["anomaly_abs"], anom=res["anomaly_pct"])
mp = v.production_t.mean()

print(f"Yillar: {len(v)} | O'rtacha: {mp:,.0f} t | CV: {v.production_t.std(ddof=1)/mp*100:.1f}%")
print(f"Trend: {res['slope']:+,.0f} t/yil, p = {res['p_value']:.4f}")
i18 = list(v.year).index(2018)
print(f"2018: {res['anomaly_abs'][i18]:,.0f} t ({res['anomaly_pct'][i18]:.1f}%)")

In [ ]:
panel = detrend_panel(load_districts())
from s06_product import attach_province_index
panel = attach_province_index(panel, v)   # viloyat HOSILDORLIK anomaliyasi
panel = panel.merge(load_climate(), on=["district","year"], how="left")
print(f"Panel: {len(panel)} kuzatuv, {panel.district.nunique()} tuman")
panel.head()

## 4. Iqlim bog'liqligi

Diqqat: yalpi korrelyatsiya nolga yaqin. Tuman va yil effektlari olib tashlangandan
keyin bog'liqlik ahamiyatli bo'ladi, ammo R² juda past — bu 3-bobdagi asosiy uslubiy
xulosaning asosi.

In [ ]:
from s03_climate import pearson_table, two_way_within, bonferroni_threshold

print(pearson_table(panel, "anom", ["heat35","heat30","precip_sum","rh_min_mean"]).to_string(index=False))
w = two_way_within(panel, "anom", "heat35")
print(f"\nIchki (within): r = {w['r']:+.3f}, p = {w['p']:.4f}, R2 = {w['r_squared']:.3f}")
print(f"Bonferroni chegarasi (6 test): {bonferroni_threshold(6):.4f}")

## 5. Barqarorlik sinovlari — chang gipotezasining falsifikatsiyasi

Jackknife jadvalida 2018-yil qatoriga e'tibor bering.

In [ ]:
from s04_robustness import jackknife_correlation, permutation_test, partial_correlation
from scipy import stats

st = load_station(); st = st[st.station == "Termiz"]
m = v[["year","anom_abs"]].merge(st[["year","dust_days","heat_days"]], on="year")
m = m[(m.year >= 2014) & (m.year <= 2024)].reset_index(drop=True)

r0, p0, jk = jackknife_correlation(m.anom_abs, m.dust_days)
print(f"Asosiy natija: r = {r0:+.3f}, p = {p0:.4f}, n = {len(m)}\n")
print(jk.assign(yil=m.year.values)[["yil","r","p","ahamiyatli"]].to_string(index=False))

In [ ]:
print(f"Spearman     : rho = {stats.spearmanr(m.anom_abs, m.dust_days)[0]:+.3f}")
pr, pp = partial_correlation(m.anom_abs, m.dust_days, m.heat_days)
print(f"Qisman (heat): r   = {pr:+.3f}, p = {pp:.4f}")
print(f"Permutatsiya : p   = {permutation_test(m.anom_abs, m.dust_days):.4f}")

## 6–8. Ekstremal qiymatlar nazariyasi

Avval GEV shakl parametrining baholanishi tekshiriladi. Ishonch oralig'i nolni va
ikkala ishorani qamrasa, parametr aniqlanmagan hisoblanadi va Gumbel tanlanadi.

In [ ]:
from s05_evt import (loss_series, fit_gumbel, check_gev_stability, gev_profile_likelihood,
                     return_levels, bootstrap_return_levels, var_cvar)

L = loss_series(v.anom_abs)
gev = check_gev_stability(L)
print(f"GEV xi = {gev['xi']:+.3f}, 95% CI = [{gev['ci_low']:+.2f}, {gev['ci_high']:+.2f}]")
print("Aniqlangan" if gev["identified"] else "ANIQLANMAGAN -> Gumbel tanlanadi")
pl = gev_profile_likelihood(L)
print(f"Profil ehtimollik: xi = {pl['xi']:+.3f}, 95% CI = [{pl['ci_low']:+.3f}, {pl['ci_high']:+.3f}], LR = {pl['LR']:.3f} (p = {pl['p']:.3f})")

g = fit_gumbel(L)
print(f"\nGumbel: mu = {g['mu']:,.0f}, sigma = {g['sigma']:,.0f}")

In [ ]:
print(return_levels(g["dist"], mean_production=mp).to_string(index=False))
print()
print(var_cvar(g["dist"], mean_production=mp).to_string(index=False))
print()
print(bootstrap_return_levels(L, mean_production=mp).to_string(index=False))

## 9–11. Mahsulot dizayni va bazis riski

Tuman va viloyat indekslarini taqqoslash bazis riskining miqdoriy o'lchovini beradi.

Yuqori xavf zonasi mezon bo'yicha aniqlanadi (anomaliya SD > 14% va viloyat bilan r < 0,5); subsidiya — mukofotning 50 foizi, chegarasiz (VM 206-son qarori, 15-ilova).

In [ ]:
from s06_product import simulate_index, summarise, basis_risk_table, zone_tariffs

for lvl, lab in [("district","Tuman indeksi"), ("province","Viloyat indeksi")]:
    print(f"{lab:18s}: {summarise(simulate_index(panel, 10.0, lvl))}")

In [ ]:
br = basis_risk_table(panel)
print(br.to_string(index=False))
t = br.iloc[-1]
print(f"\nQoplash darajasi: {t.qoplangan}/{t.zarar_yillari} = {t.qoplangan/t.zarar_yillari*100:.0f}%")

In [ ]:
zt = zone_tariffs(panel)
print(zt.to_string(index=False))

## 12. Chizmalar

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].bar(v.year, v.anom_abs/1000, color=["#c0392b" if a < -50000 else "#2c5f8d" for a in v.anom_abs])
ax[0].axhline(0, color="k", lw=0.8)
ax[0].set_xlabel("Yil"); ax[0].set_ylabel("Anomaliya, ming tonna")
ax[0].set_title("Trenddan og'ish")

xs = np.linspace(L.min()-20000, L.max()+40000, 400)
ax[1].hist(L, bins=8, density=True, color="#aac4dd", edgecolor="white")
ax[1].plot(xs, g["dist"].pdf(xs), color="#c0392b", lw=2, label="Gumbel")
ax[1].axvline(L.max(), color="#c0392b", ls="--", lw=1.2, label="2018")
ax[1].set_xlabel("Yo'qotish, tonna"); ax[1].set_ylabel("Zichlik")
ax[1].set_title("Yo'qotishlar taqsimoti"); ax[1].legend()
plt.tight_layout(); plt.show()

## 13. Butun zanjirni bir buyruq bilan

Yuqoridagi barcha bosqichlarni qayta hisoblab, natijalarni `outputs/` ga yozadi.

In [ ]:
!python run_all.py

## Qo'shimcha dissertatsiya jadvallari

2.2.5, 2.3.1 (15 yillik oyna), 2.3.3, 2.4.1–2.4.3, 3.1.2 va 3.1.4-jadvallar.

In [ ]:
from s07_tables import (panel_fe_table, effect_size, window_evt, evt_tariffs,
                       district_table, province_anomaly_sd, simulate_heat_index, deductible_table)
fe = panel_fe_table(panel); print(fe.to_string(index=False))
print(effect_size(panel, abs(fe.loc[1, 'beta_2018siz'])))
print(district_table(panel).to_string(index=False))
print(f"Viloyat anomaliyasi SD: {province_anomaly_sd(v)}%")
print(simulate_heat_index(panel).to_string(index=False))
print(deductible_table(panel).to_string(index=False))